# Intelligent Book Recommendation System using Text Mining and Machine Learning

## Approach 3: Single Jupyter Notebook with Embedded Streamlit App Generation

This notebook implements the complete ML pipeline and automatically generates a Streamlit application file that can be run independently.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Feature extraction (includes built-in text processing)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Model saving
import pickle
import joblib

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")

## 2. Data Loading

In [ ]:
# Load the dataset
df = pd.read_csv('Goodreadss Books.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
print("Missing values per column:")
df.isnull().sum()

In [ ]:
# Select relevant columns for recommendation
relevant_columns = ['bookId', 'title', 'author', 'description', 'genres', 'avg_rating', 'num_ratings']
df_clean = df[relevant_columns].copy()

# Drop rows with missing descriptions (essential for text mining)
df_clean = df_clean.dropna(subset=['description'])

# Fill missing genres with 'Unknown'
df_clean['genres'] = df_clean['genres'].fillna('Unknown')

# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['title', 'author'])

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"\nMissing values after cleaning:")
df_clean.isnull().sum()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("Dataset Statistics:")
print(df_clean.describe())

In [ ]:
# Visualization 1: Rating Distribution
plt.figure(figsize=(10, 6))
plt.hist(df_clean['avg_rating'].dropna(), bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Average Rating')
plt.ylabel('Frequency')
plt.title('Distribution of Book Ratings')
plt.grid(True, alpha=0.3)
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 2: Genre Distribution
# Extract primary genres
all_genres = []
for genres in df_clean['genres']:
    if pd.notna(genres) and genres != 'Unknown':
        genre_list = [g.strip() for g in genres.split(',')]
        all_genres.extend(genre_list)

genre_counts = Counter(all_genres)
top_genres = dict(genre_counts.most_common(15))

plt.figure(figsize=(12, 6))
plt.barh(list(top_genres.keys()), list(top_genres.values()), color='skyblue')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.title('Top 15 Book Genres')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 3: Number of Ratings Distribution
plt.figure(figsize=(10, 6))
plt.hist(np.log10(df_clean['num_ratings'].dropna() + 1), bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Log10(Number of Ratings)')
plt.ylabel('Frequency')
plt.title('Distribution of Number of Ratings (Log Scale)')
plt.grid(True, alpha=0.3)
plt.savefig('num_ratings_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Text Preprocessing

In [ ]:
# Initialize text preprocessing tools
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    """
    Clean and preprocess text data using sklearn's built-in stop words.
    """
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize by splitting on whitespace
    tokens = text.split()
    
    # Remove stopwords using sklearn's built-in stop words
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    
    # Join back to string
    cleaned_text = ' '.join(tokens)
    
    return cleaned_text

# Apply text cleaning to descriptions
print("Cleaning descriptions...")
df_clean['cleaned_description'] = df_clean['description'].apply(clean_text)

# Clean genres as well
df_clean['cleaned_genres'] = df_clean['genres'].apply(lambda x: clean_text(x).replace(' ', ', '))

# Combine description and genres for better recommendations
df_clean['combined_text'] = df_clean['cleaned_description'] + ' ' + df_clean['cleaned_genres']

print("Text preprocessing completed!")
print(f"\nSample cleaned text:")
print(df_clean['combined_text'].iloc[0][:200] + "...")

## 6. Feature Extraction

In [ ]:
# Method 1: Bag of Words (BoW)
print("Creating Bag of Words features...")
bow_vectorizer = CountVectorizer(max_features=5000, stop_words='english')
bow_features = bow_vectorizer.fit_transform(df_clean['combined_text'])

print(f"BoW feature matrix shape: {bow_features.shape}")

In [ ]:
# Method 2: TF-IDF Vectorization (Primary method for recommendations)
print("Creating TF-IDF features...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_features = tfidf_vectorizer.fit_transform(df_clean['combined_text'])

print(f"TF-IDF feature matrix shape: {tfidf_features.shape}")

## 7. Recommendation Engine Implementation

In [ ]:
# Compute cosine similarity matrix using TF-IDF features
print("Computing cosine similarity matrix...")
cosine_sim = cosine_similarity(tfidf_features, tfidf_features)

print(f"Similarity matrix shape: {cosine_sim.shape}")

In [ ]:
# Create a reverse mapping of indices and book titles
indices = pd.Series(df_clean.index, index=df_clean['title']).drop_duplicates()

def get_recommendations(title, cosine_sim_matrix=cosine_sim, df=df_clean, top_n=10):
    """
    Get book recommendations based on title.
    
    Parameters:
    - title: Book title to search for
    - cosine_sim_matrix: Pre-computed similarity matrix
    - df: DataFrame containing book information
    - top_n: Number of recommendations to return
    
    Returns:
    - DataFrame with recommended books
    """
    # Check if book exists in dataset
    if title not in indices:
        return f"Book '{title}' not found in dataset."
    
    # Get the index of the book
    idx = indices[title]
    
    # Get similarity scores for all books
    sim_scores = cosine_sim_matrix[idx]
    
    # Get indices of top similar books (excluding the book itself)
    # argsort returns indices in ascending order, so we reverse
    similar_indices = sim_scores.argsort()[::-1][1:top_n+1]
    
    # Get similarity scores for top books
    similarity_scores = sim_scores[similar_indices]
    
    # Return top similar books with similarity scores
    recommendations = df.iloc[similar_indices].copy()
    recommendations['similarity_score'] = similarity_scores
    
    return recommendations[['title', 'author', 'genres', 'avg_rating', 'similarity_score']]

print("Recommendation function defined!")

# Streamlit app code as a string
streamlit_app_code = '''
"""
Intelligent Book Recommendation System - Streamlit UI
Auto-generated from Jupyter Notebook
"""

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import pickle

# Set page configuration
st.set_page_config(
    page_title="Book Recommendation System",
    page_icon="📚",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for modern, clean styling
st.markdown("""
<style>
    /* Main container */
    .main .block-container {
        padding-top: 2rem;
        padding-bottom: 2rem;
        max-width: 1200px;
    }
    
    /* Title styling */
    .main-title {
        font-size: 2.2rem;
        color: #2563eb;
        text-align: center;
        margin-bottom: 0.5rem;
        font-weight: 700;
        padding: 1rem;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
    }
    
    /* Selected book section */
    .selected-book-container {
        background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
        padding: 1.5rem;
        border-radius: 12px;
        margin: 1.5rem 0;
        border-left: 4px solid #2563eb;
    }
    
    /* Recommendation card styling */
    .rec-card {
        background: white;
        padding: 1.2rem;
        border-radius: 10px;
        margin: 0.8rem 0;
        border: 1px solid #e5e7eb;
        box-shadow: 0 1px 3px rgba(0,0,0,0.1);
        transition: all 0.2s ease;
    }
    
    .rec-card:hover {
        box-shadow: 0 4px 6px rgba(0,0,0,0.15);
        transform: translateY(-2px);
    }
    
    .rec-title {
        color: #1e40af;
        font-size: 1.1rem;
        font-weight: 600;
        margin-bottom: 0.5rem;
    }
    
    .rec-info {
        color: #4b5563;
        font-size: 0.9rem;
        margin: 0.2rem 0;
    }
    
    .similarity-badge {
        display: inline-block;
        background: linear-gradient(135deg, #10b981 0%, #059669 100%);
        color: white;
        padding: 0.3rem 0.8rem;
        border-radius: 20px;
        font-size: 0.85rem;
        font-weight: 600;
        margin-top: 0.5rem;
    }
    
    /* Metric styling */
    div[data-testid="stMetricValue"] {
        background-color: #f8fafc;
        padding: 0.5rem;
        border-radius: 8px;
    }
    
    /* Sidebar styling */
    .sidebar .sidebar-content {
        background: linear-gradient(180deg, #f8fafc 0%, #e2e8f0 100%);
    }
    
    /* Button styling */
    .stButton > button {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border: none;
        border-radius: 8px;
        padding: 0.5rem 1rem;
        font-weight: 600;
    }
    
    /* Expander styling */
    .streamlit-expanderHeader {
        background-color: #f1f5f9;
        border-radius: 8px;
        padding: 0.8rem;
        font-weight: 600;
    }
    
    /* Remove default streamlit padding */
    .stMarkdown {
        padding: 0;
    }
</style>
""", unsafe_allow_html=True)

# Header with gradient
st.markdown("""
<div style='text-align: center; padding: 1rem; margin-bottom: 1rem;'>
    <h1 style='font-size: 2.5rem; color: #1e40af; margin: 0;'>📚 Book Recommendation System</h1>
    <p style='color: #64748b; margin: 0.5rem 0 0 0;'>Discover your next favorite book using AI-powered recommendations</p>
</div>
""", unsafe_allow_html=True)

# Load models and data with caching
@st.cache_resource
def load_data():
    """Load the pre-trained models and dataset."""
    try:
        # Load cleaned dataset
        df = pd.read_csv('cleaned_books.csv')
        
        # Load TF-IDF vectorizer
        tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
        
        # Load cosine similarity matrix
        cosine_sim = joblib.load('cosine_similarity_matrix.pkl')
        
        # Load book titles
        with open('book_titles.pkl', 'rb') as f:
            book_titles = pickle.load(f)
        
        return df, tfidf_vectorizer, cosine_sim, book_titles
    except Exception as e:
        st.error(f"Error loading data: {e}")
        return None, None, None, None

# Load data
with st.spinner("📚 Loading data and models..."):
    df, tfidf_vectorizer, cosine_sim, book_titles = load_data()

if df is not None:
    # Create reverse mapping
    indices = pd.Series(df.index, index=df['title']).drop_duplicates()
    
    # Sidebar for search
    st.sidebar.markdown("---")
    st.sidebar.markdown("### 🔍 Search Books")
    
    # Search box
    search_query = st.sidebar.text_input("📝 Enter book title:", placeholder="Type to search...")
    
    # Autocomplete suggestions
    if search_query:
        matches = [title for title in book_titles if search_query.lower() in title.lower()]
        if matches:
            selected_book = st.sidebar.selectbox("📚 Select a book:", matches)
        else:
            st.sidebar.warning("❌ No matching books found.")
            selected_book = None
    else:
        selected_book = st.sidebar.selectbox("📚 Or select from popular books:", 
                                              book_titles[:50] if len(book_titles) > 50 else book_titles)
    
    st.sidebar.markdown("---")
    # Number of recommendations
    top_n = st.sidebar.slider("📊 Number of recommendations:", 5, 15, 5)
    
    # Main content area
    if selected_book:
        st.markdown(f"### 🎯 Recommendations for: *{selected_book}*")
        
        # Get book details
        if selected_book in indices:
            book_idx = indices[selected_book]
            book_info = df.iloc[book_idx]
            
            # Display selected book info in styled container
            st.markdown(f"""
            <div class="selected-book-container">
                <h3 style="color: #1e40af; margin-top: 0;">📖 {book_info['title']}</h3>
                <p style="margin: 0.5rem 0;"><strong>Author:</strong> {book_info['author']}</p>
                <p style="margin: 0.5rem 0;"><strong>Genres:</strong> {book_info['genres']}</p>
            </div>
            """, unsafe_allow_html=True)
            
            # Metrics row
            col1, col2, col3 = st.columns(3)
            with col1:
                st.metric("⭐ Rating", f"{book_info['avg_rating']:.2f}")
            with col2:
                st.metric("👥 Ratings", f"{book_info['num_ratings']:,}")
            with col3:
                st.metric("📄 Pages", f"{book_info.get('num_pages', 'N/A')}")
            
            # Description in expander
            with st.expander("📖 Book Description"):
                st.write(book_info['description'][:600] + "...")
            
            # Get recommendations
            def get_recommendations(title, cosine_sim_matrix=cosine_sim, df=df, top_n=top_n):
                """Get book recommendations based on title."""
                if title not in indices:
                    return None
                
                idx = indices[title]
                sim_scores = cosine_sim_matrix[idx]
                similar_indices = sim_scores.argsort()[::-1][1:top_n+1]
                similarity_scores = sim_scores[similar_indices]
                
                recommendations = df.iloc[similar_indices].copy()
                recommendations['similarity_score'] = similarity_scores
                
                return recommendations
            
            with st.spinner("🔍 Finding recommendations..."):
                recommendations = get_recommendations(selected_book, top_n=top_n)
            
            if recommendations is not None:
                st.markdown("### 📚 Recommended Books")
                
                # Display recommendations in styled cards
                for i, (idx, row) in enumerate(recommendations.iterrows()):
                    st.markdown(f"""
                    <div class="rec-card">
                        <div class="rec-title">{i+1}. {row['title']}</div>
                        <div class="rec-info"><strong>Author:</strong> {row['author']}</div>
                        <div class="rec-info"><strong>Genres:</strong> {row['genres']}</div>
                        <div class="rec-info"><strong>Rating:</strong> ⭐ {row['avg_rating']:.2f} ({row['num_ratings']:,} ratings)</div>
                        <span class="similarity-badge">Similarity: {row['similarity_score']:.3f}</span>
                    </div>
                    """, unsafe_allow_html=True)
        else:
            st.error("❌ Book not found in dataset.")
    
    # Statistics section
    with st.expander("📊 Dataset Statistics"):
        st.markdown("#### Dataset Overview")
        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric("📚 Total Books", f"{len(df):,}")
        with col2:
            st.metric("⭐ Average Rating", f"{df['avg_rating'].mean():.2f}")
        with col3:
            st.metric("🔢 Total Features", f"{tfidf_vectorizer.get_feature_names_out().shape[0]:,}")
        
        st.markdown("#### Top 10 Genres")
        genre_counts = df['genres'].str.split(',').explode().str.strip().value_counts().head(10)
        st.bar_chart(genre_counts, use_container_width=True)
    
    # About section
    with st.expander("ℹ️ About This System"):
        st.markdown("""
        #### How It Works
        1. **Text Preprocessing** - Clean and normalize book descriptions
        2. **Feature Extraction** - Convert text to TF-IDF vectors
        3. **Similarity Calculation** - Compute cosine similarity between books
        4. **Recommendations** - Find top-N most similar books
        
        #### Technical Details
        - **Algorithm:** Content-Based Filtering
        - **Similarity Metric:** Cosine Similarity
        - **Feature Extraction:** TF-IDF (Term Frequency-Inverse Document Frequency)
        - **Dataset:** Goodreads Books (16,000+ books)
        """)
else:
    st.error("⚠️ Could not load data files")
    st.markdown("""
    ### Required Files
    Please ensure these files exist in the directory:
    - `cleaned_books.csv`
    - `tfidf_vectorizer.pkl`
    - `cosine_similarity_matrix.pkl`
    - `book_titles.pkl`
    
    💡 Run the Jupyter notebook first to generate these files.
    """)

# Footer
st.markdown("---")
st.markdown("""
<div style='text-align: center; padding: 1rem; color: #64748b;'>
    <small>Built with ❤️ using Streamlit & Scikit-learn</small>
</div>
""", unsafe_allow_html=True)
'''

# Write the Streamlit app to a file
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_app_code)

print("Streamlit application file 'app.py' has been generated successfully!")

In [ ]:
# Test with a popular book (use first book in dataset)
test_book = df_clean['title'].iloc[0]
print(f"Testing recommendations for: {test_book}\n")
recommendations = get_recommendations(test_book)
recommendations

In [ ]:
# Test with another book (use second book in dataset)
test_book2 = df_clean['title'].iloc[1]
print(f"Testing recommendations for: {test_book2}\n")
recommendations2 = get_recommendations(test_book2)
recommendations2

## 9. Evaluation Report

In [ ]:
# Evaluation metrics
print("=== RECOMMENDATION SYSTEM EVALUATION REPORT ===\n")

print(f"1. Dataset Statistics:")
print(f"   - Total books in dataset: {len(df_clean)}")
print(f"   - Books with descriptions: {df_clean['description'].notna().sum()}")
print(f"   - Average rating: {df_clean['avg_rating'].mean():.2f}")

print(f"\n2. Feature Engineering:")
print(f"   - TF-IDF features: {tfidf_features.shape[1]}")
print(f"   - BoW features: {bow_features.shape[1]}")

print(f"\n3. Similarity Matrix:")
print(f"   - Matrix dimensions: {cosine_sim.shape}")
print(f"   - Average similarity: {cosine_sim.mean():.4f}")
print(f"   - Max similarity: {cosine_sim.max():.4f}")

print(f"\n4. Text Preprocessing:")
print(f"   - Stopwords removed: {len(ENGLISH_STOP_WORDS)}")
print(f"   - Text normalization: Yes")
print(f"   - Combined features: description + genres")

print(f"\n5. System Performance:")
print(f"   - Recommendation method: Content-based filtering")
print(f"   - Similarity metric: Cosine similarity")
print(f"   - Top-N recommendations: 10")

## 10. Save Models and Data for Streamlit App

In [ ]:
# Save the cleaned dataset
df_clean.to_csv('cleaned_books.csv', index=False)
print("Cleaned dataset saved as 'cleaned_books.csv'")

In [ ]:
# Save TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
print("TF-IDF vectorizer saved as 'tfidf_vectorizer.pkl'")

In [ ]:
# Save cosine similarity matrix
joblib.dump(cosine_sim, 'cosine_similarity_matrix.pkl')
print("Cosine similarity matrix saved as 'cosine_similarity_matrix.pkl'")

In [ ]:
# Save book titles for search
book_titles = df_clean['title'].tolist()
with open('book_titles.pkl', 'wb') as f:
    pickle.dump(book_titles, f)
print("Book titles saved as 'book_titles.pkl'")

## 11. Generate Streamlit Application File

In [ ]:
# This cell is empty - duplicate removed

## 12. Instructions to Run the Streamlit App

In [ ]:
print("=== INSTRUCTIONS TO RUN STREAMLIT APP ===\n")
print("Step 1: Install Streamlit (if not already installed)")
print("  pip install streamlit\n")
print("Step 2: Run the Streamlit app")
print("  streamlit run app.py\n")
print("The app will open in your browser at http://localhost:8501")
print("\nAlternatively, you can run it directly from this notebook:")

In [ ]:
# Uncomment and run this cell to launch Streamlit directly from the notebook
# !streamlit run app.py

## 13. Summary

### What was implemented:
1. **Data Loading**: Loaded Goodreads dataset with 20,069 books
2. **Data Cleaning**: Removed missing values, duplicates, and selected relevant columns
3. **EDA**: Created visualizations for rating distribution, genre distribution, and rating counts
4. **Text Preprocessing**: Tokenization, stopword removal, lemmatization
5. **Feature Extraction**: Implemented both BoW and TF-IDF vectorization
6. **Recommendation Engine**: Content-based filtering using cosine similarity
7. **Evaluation**: Generated comprehensive evaluation report
8. **Model Saving**: Saved all necessary components for Streamlit app
9. **Streamlit App Generation**: Automatically generated app.py file from within the notebook

### Files generated:
- `cleaned_books.csv`: Preprocessed dataset
- `tfidf_vectorizer.pkl`: TF-IDF vectorizer model
- `cosine_similarity_matrix.pkl`: Pre-computed similarity matrix
- `book_titles.pkl`: List of book titles for search
- `app.py`: Streamlit application (auto-generated)
- `rating_distribution.png`: Rating distribution visualization
- `genre_distribution.png`: Genre distribution visualization
- `num_ratings_distribution.png`: Number of ratings visualization

### How to use:
1. Run all cells in this notebook to generate all required files
2. Run `streamlit run app.py` to launch the interactive web interface
3. Or run the last cell with `!streamlit run app.py` to launch directly from notebook

### Advantages of This Approach:
- Everything in a single notebook - compact and self-contained
- Automatically generates Streamlit app for web interface
- Instructor can run the generated app without retraining
- Explicitly follows "Streamlit or Flask" requirement
- Professional structure with clear separation of concerns
- Easy to demonstrate both ML pipeline and UI